BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & CẤU HÌNH SIÊU TỐC

In [15]:
# Sửa lỗi xung đột Protobuf
!pip uninstall -y protobuf
!pip install protobuf==3.20.3
!pip install transformers --upgrade

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is inc

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
!pip install tf-keras

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
# [BLOCK 1 - CẤU HÌNH TURBO TĂNG TỐC]
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Bắt buộc cho Kaggle mới

import pandas as pd
import numpy as np
import tensorflow as tf
import re
import gc
from transformers import RobertaTokenizerFast, TFRobertaModel
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.mixed_precision import set_global_policy

set_global_policy('mixed_float16') # Bật chế độ tiết kiệm RAM

# --- THAY ĐỔI ĐỂ TĂNG TỐC GẤP 5 LẦN ---
MAX_LEN = 256      # Giảm độ dài (Đủ để đoán tính cách rồi)
BATCH_SIZE = 32    # Tăng batch size (Xử lý nhiều hơn 1 lúc)
EPOCHS = 2         # Giảm xuống 2 vòng (Data lớn 2 vòng là đủ khôn)
LEARNING_RATE = 1e-5
# --------------------------------------

print(f"TensorFlow Version: {tf.__version__}")
print("✅ ĐÃ BẬT CHẾ ĐỘ TĂNG TỐC (TURBO MODE)!")

TensorFlow Version: 2.19.0
✅ ĐÃ BẬT CHẾ ĐỘ TĂNG TỐC (TURBO MODE)!


BƯỚC 2: XỬ LÝ DỮ LIỆU & TOKENIZER

In [18]:
# [BLOCK 2 - CẮT GỌT DỮ LIỆU MỚI]
# Phần tìm file giữ nguyên, ta tận dụng biến DATA_PATH cũ (nếu có)
# Nếu mất biến DATA_PATH thì chạy lại code tìm file, còn không thì chạy thẳng đoạn này:

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

# Hàm làm sạch
def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                  'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    return re.sub(pattern, '', text).strip()

# Hàm cắt gọt mới (Cho MAX_LEN 256)
def smart_truncate(text, max_words=230): 
    words = text.split()
    if len(words) <= max_words: return text
    # Lấy 100 từ đầu + 130 từ cuối (Vẫn đảm bảo nắm ý chính)
    return " ".join(words[:100] + words[-130:])

print("-> Đang xử lý lại dữ liệu (Nhanh thôi)...")
# Load lại để đảm bảo sạch sẽ
if 'DATA_PATH' not in globals():
    # Phòng hờ bị mất biến, tìm lại nhanh
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if '500' in filename: DATA_PATH = os.path.join(dirname, filename); break

df = pd.read_csv(DATA_PATH)
df['clean_text'] = df['posts'].apply(basic_clean)
df['smart_text'] = df['clean_text'].apply(smart_truncate)
print("✅ Xử lý xong! Dữ liệu đã gọn nhẹ.")

-> Đang xử lý lại dữ liệu (Nhanh thôi)...
✅ Xử lý xong! Dữ liệu đã gọn nhẹ.


BƯỚC 3: XÂY DỰNG KIẾN TRÚC MODEL

In [19]:
# [BLOCK 3] Kiến trúc Model
def build_roberta_model():
    input_ids = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input_ids')
    mask = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='attention_mask')
    
    # QUAN TRỌNG: use_safetensors=False để tránh lỗi tải file
    roberta = TFRobertaModel.from_pretrained('roberta-base', use_safetensors=False)
    
    embeddings = roberta(input_ids, attention_mask=mask)[0]
    cls_token = embeddings[:, 0, :]
    
    x = tf.keras.layers.Dense(256, activation='relu')(cls_token)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    
    model = tf.keras.Model(inputs=[input_ids, mask], outputs=output)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def create_dataset(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='tf')
    dataset = tf.data.Dataset.from_tensor_slices((dict(encodings), labels))
    return dataset.shuffle(2048).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ Đã dựng xong hàm Model (Fix SafeTensors)!")

✅ Đã dựng xong hàm Model (Fix SafeTensors)!


BƯỚC 4: HUẤN LUYỆN TOÀN DIỆN (4 TRỤC)

In [8]:
# [BLOCK 4] Chạy Training & Lưu báo cáo
if not os.path.exists('models'): os.makedirs('models')
final_results = []

axes = {'IE': ('I', 'E'), 'NS': ('N', 'S'), 'TF': ('F', 'T'), 'JP': ('J', 'P')}

print("🚀 BẮT ĐẦU CHIẾN DỊCH HUẤN LUYỆN...")

for axis, (char0, char1) in axes.items():
    print(f"\n🔥 TRỤC: {axis} ({char0} vs {char1})")
    
    # Gán nhãn & Chia tập
    df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['smart_text'].tolist(), df['label'].tolist(), test_size=0.1, random_state=42, stratify=df['label']
    )
    
    # Tính toán trọng số lớp (Cân bằng dữ liệu)
    weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights = {0: weights[0], 1: weights[1]}
    
    # Train
    tf.keras.backend.clear_session()
    model = build_roberta_model()
    
    checkpoint_path = f"models/best_model_{axis}.h5"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_loss', save_best_only=True, save_weights_only=True, mode='min', verbose=1
    )
    
    history = model.fit(
        create_dataset(train_texts, train_labels),
        validation_data=create_dataset(val_texts, val_labels),
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=[checkpoint]
    )
    
    # Ghi lại kết quả
    best_acc = max(history.history['val_accuracy'])
    final_results.append({'Axis': axis, 'Best Accuracy': best_acc})
    print(f"✅ Xong trục {axis}! Acc: {best_acc:.4f}")
    
    del model
    gc.collect()

# Xuất báo cáo
pd.DataFrame(final_results).to_csv('final_report.csv', index=False)
tokenizer.save_pretrained('models/roberta_tokenizer')
print("\n🎉 HOÀN TẤT TOÀN BỘ! ĐÃ LƯU FILE 'final_report.csv'.")

IndentationError: unexpected indent (2273744207.py, line 2)

Giữa chừng bị mất kết nối nên lưu lại 2 cặp nhãn đã huấn luyện xong!

In [7]:
import os
from IPython.display import FileLink

# 1. Kiểm tra xem file có còn sống sót không
print("🔍 ĐANG TÌM KIẾM FILE MODEL...")
files = os.listdir("models")
if len(files) > 0:
    print(f"✅ QUÁ MAY MẮN! Tìm thấy {len(files)} file trong thư mục models:")
    print(files)
else:
    print("❌ CHIA BUỒN: Thư mục rỗng. Có thể Session đã bị reset hoàn toàn.")

# 2. Tạo link tải về (Nếu có file)
print("\n⬇️ BẤM VÀO LINK DƯỚI ĐỂ TẢI VỀ NGAY (ĐỪNG CHẦN CHỪ):")
for f in files:
    if f.endswith('.h5'):
        print(f"👉 Tải {f}:")
        display(FileLink(f'models/{f}'))

🔍 ĐANG TÌM KIẾM FILE MODEL...
✅ QUÁ MAY MẮN! Tìm thấy 2 file trong thư mục models:
['best_model_NS.h5', 'best_model_IE.h5']

⬇️ BẤM VÀO LINK DƯỚI ĐỂ TẢI VỀ NGAY (ĐỪNG CHẦN CHỪ):
👉 Tải best_model_NS.h5:


/kaggle/working/models/best_model_NS.h5

👉 Tải best_model_IE.h5:


/kaggle/working/models/best_model_IE.h5

Xuất metrics của cặp I/E và N/S

In [10]:
# [BLOCK ĐÁNH GIÁ CHUYÊN SÂU]
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import re
from transformers import RobertaTokenizerFast, TFRobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. CẤU HÌNH (Phải khớp với lúc Train Turbo)
MAX_LEN = 256
BATCH_SIZE = 32

# 2. CHUẨN BỊ DỮ LIỆU (NẾU MẤT BIẾN)
if 'df' not in globals() or 'smart_text' not in df.columns:
    print("⏳ Đang nạp lại dữ liệu...")
    # Tìm file
    data_path = None
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if '500' in filename: data_path = os.path.join(dirname, filename); break
    
    df = pd.read_csv(data_path)
    tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
    
    def basic_clean(text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\.\S+', '', text)
        mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                      'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
        pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
        return re.sub(pattern, '', text).strip()

    def smart_truncate(text, max_words=230):
        words = text.split()
        if len(words) <= max_words: return text
        return " ".join(words[:100] + words[-130:])

    df['clean_text'] = df['posts'].apply(basic_clean)
    df['smart_text'] = df['clean_text'].apply(smart_truncate)
    print("✅ Xử lý dữ liệu xong.")

# 3. HÀM DỰNG MODEL & DATASET
def build_roberta_model():
    input_ids = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input_ids')
    mask = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='attention_mask')
    roberta = TFRobertaModel.from_pretrained('roberta-base', use_safetensors=False)
    embeddings = roberta(input_ids, attention_mask=mask)[0]
    cls_token = embeddings[:, 0, :]
    x = tf.keras.layers.Dense(256, activation='relu')(cls_token)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs=[input_ids, mask], outputs=output)
    return model

def create_dataset(texts):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='tf')
    # Chỉ trả về input (không cần label) để predict
    return tf.data.Dataset.from_tensor_slices(dict(encodings)).batch(BATCH_SIZE)

# 4. TÍNH TOÁN CHỈ SỐ
axes_to_check = {'IE': ('I', 'E'), 'NS': ('N', 'S')}

print("\n" + "="*60)
print("📊 BÁO CÁO HIỆU SUẤT MODEL (ACCURACY - MACRO F1 - WEIGHTED F1)")
print("="*60)

results_summary = []

for axis, (char0, char1) in axes_to_check.items():
    model_path = f"models/best_model_{axis}.h5"
    
    if os.path.exists(model_path):
        print(f"\n🔍 Đang đánh giá trục: {axis}...")
        
        # Tái tạo tập validation
        df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)
        _, val_texts, _, val_labels = train_test_split(
            df['smart_text'].tolist(), df['label'].tolist(), test_size=0.1, random_state=42, stratify=df['label']
        )
        
        # Load Model
        model = build_roberta_model()
        model.load_weights(model_path)
        
        # Predict
        print("   -> Đang dự đoán...")
        y_prob = model.predict(create_dataset(val_texts), verbose=0)
        y_pred = (y_prob > 0.5).astype(int).flatten()
        
        # Tính chỉ số
        acc = accuracy_score(val_labels, y_pred)
        mf1 = f1_score(val_labels, y_pred, average='macro')
        wf1 = f1_score(val_labels, y_pred, average='weighted')
        
        print(f"✅ KẾT QUẢ TRỤC {axis}:")
        print(f"   - Accuracy:    {acc:.4f}")
        print(f"   - Macro F1:    {mf1:.4f}  (Quan trọng cho lớp thiểu số)")
        print(f"   - Weighted F1: {wf1:.4f}  (Quan trọng tổng thể)")
        
        results_summary.append({'Axis': axis, 'Accuracy': acc, 'Macro F1': mf1, 'Weighted F1': wf1})
        
        # Dọn dẹp
        del model
        tf.keras.backend.clear_session()
        gc.collect()
    else:
        print(f"❌ Không tìm thấy file {model_path}")

# In bảng tổng kết
print("\n" + "="*60)
print("🏆 TỔNG HỢP KẾT QUẢ")
final_df = pd.DataFrame(results_summary)
pd.options.display.float_format = '{:.4f}'.format
print(final_df)
print("="*60)


📊 BÁO CÁO HIỆU SUẤT MODEL (ACCURACY - MACRO F1 - WEIGHTED F1)

🔍 Đang đánh giá trục: IE...


Some layers from the model checkpoint at roberta-base were not used when initializing TFRobertaModel: ['lm_head']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at roberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


   -> Đang dự đoán...
✅ KẾT QUẢ TRỤC IE:
   - Accuracy:    0.8052
   - Macro F1:    0.7472  (Quan trọng cho lớp thiểu số)
   - Weighted F1: 0.8103  (Quan trọng tổng thể)

🔍 Đang đánh giá trục: NS...


Some layers from the model checkpoint at roberta-base were not used when initializing TFRobertaModel: ['lm_head']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at roberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


   -> Đang dự đoán...
✅ KẾT QUẢ TRỤC NS:
   - Accuracy:    0.8888
   - Macro F1:    0.7252  (Quan trọng cho lớp thiểu số)
   - Weighted F1: 0.9004  (Quan trọng tổng thể)

🏆 TỔNG HỢP KẾT QUẢ
  Axis  Accuracy  Macro F1  Weighted F1
0   IE    0.8052    0.7472       0.8103
1   NS    0.8888    0.7252       0.9004


Train tiếp 2 cặp TF & JP

In [20]:
# [BLOCK 4 - ĐÃ CHỈNH SỬA] CHỈ TRAIN 2 CẶP CUỐI (TF VÀ JP)
import os
import gc
import numpy as np
import tensorflow as tf
import pandas as pd # Import thêm pandas để xuất báo cáo
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import FileLink

if not os.path.exists('models'): os.makedirs('models')
final_results = []

# --- THAY ĐỔI Ở ĐÂY: CHỈ GIỮ LẠI TF VÀ JP ---
axes = {
    # 'IE': ('I', 'E'),  <-- Đã xong, bỏ qua
    # 'NS': ('N', 'S'),  <-- Đã xong, bỏ qua
    'TF': ('F', 'T'), 
    'JP': ('J', 'P')
}
# ---------------------------------------------

print("🚀 BẮT ĐẦU CHIẾN DỊCH (CHỈ TRAIN TF VÀ JP)...")

for axis, (char0, char1) in axes.items():
    print(f"\n🔥 TRỤC: {axis} ({char0} vs {char1})")
    
    # Gán nhãn & Chia tập
    df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['smart_text'].tolist(), df['label'].tolist(), test_size=0.1, random_state=42, stratify=df['label']
    )
    
    # Tính toán trọng số lớp
    weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights = {0: weights[0], 1: weights[1]}
    
    # Reset Session cũ & Build Model
    tf.keras.backend.clear_session()
    gc.collect()
    model = build_roberta_model()
    
    # Checkpoint
    checkpoint_path = f"models/best_model_{axis}.h5"
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1
    )
    
    # Train
    history = model.fit(
        create_dataset(train_texts, train_labels),
        validation_data=create_dataset(val_texts, val_labels),
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=[checkpoint]
    )
    
    # Ghi lại kết quả
    best_acc = max(history.history['val_accuracy'])
    final_results.append({'Axis': axis, 'Best Accuracy': best_acc})
    print(f"✅ Xong trục {axis}! Acc: {best_acc:.4f}")
    
    del model
    gc.collect()

# Xuất báo cáo và Link tải
print("\n" + "="*60)
print("🎉 HOÀN TẤT TF VÀ JP! TẢI FILE NGAY DƯỚI ĐÂY:")
if os.path.exists('models/best_model_TF.h5'): 
    print("👉 Tải TF:")
    display(FileLink('models/best_model_TF.h5'))
if os.path.exists('models/best_model_JP.h5'): 
    print("👉 Tải JP:")
    display(FileLink('models/best_model_JP.h5'))

# Lưu tokenizer (cần cho lúc dùng sau này)
tokenizer.save_pretrained('models/roberta_tokenizer')

🚀 BẮT ĐẦU CHIẾN DỊCH (CHỈ TRAIN TF VÀ JP)...

🔥 TRỤC: TF (F vs T)


Some layers from the model checkpoint at roberta-base were not used when initializing TFRobertaModel: ['lm_head']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at roberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


Epoch 1/2
2984/2984 [==============================] - ETA: 0s - loss: 0.4139 - accuracy: 0.8127
Epoch 1: val_accuracy improved from -inf to 0.85246, saving model to models/best_model_TF.h5
2984/2984 [==============================] - 2967s 973ms/step - loss: 0.4139 - accuracy: 0.8127 - val_loss: 0.3447 - val_accuracy: 0.8525
Epoch 2/2
2984/2984 [==============================] - ETA: 0s - loss: 0.3311 - accuracy: 0.8571
Epoch 2: val_accuracy did not improve from 0.85246
2984/2984 [==============================] - 2853s 956ms/step - loss: 0.3311 - accuracy: 0.8571 - val_loss: 0.3588 - val_accuracy: 0.8410
✅ Xong trục TF! Acc: 0.8525

🔥 TRỤC: JP (J vs P)


Some layers from the model checkpoint at roberta-base were not used when initializing TFRobertaModel: ['lm_head']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at roberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


Epoch 1/2
2984/2984 [==============================] - ETA: 0s - loss: 0.5708 - accuracy: 0.6965
Epoch 1: val_accuracy improved from -inf to 0.73404, saving model to models/best_model_JP.h5
2984/2984 [==============================] - 2977s 974ms/step - loss: 0.5708 - accuracy: 0.6965 - val_loss: 0.5181 - val_accuracy: 0.7340
Epoch 2/2
2984/2984 [==============================] - ETA: 0s - loss: 0.5091 - accuracy: 0.7432
Epoch 2: val_accuracy improved from 0.73404 to 0.75130, saving model to models/best_model_JP.h5
2984/2984 [==============================] - 2853s 956ms/step - loss: 0.5091 - accuracy: 0.7432 - val_loss: 0.4931 - val_accuracy: 0.7513
✅ Xong trục JP! Acc: 0.7513

🎉 HOÀN TẤT TF VÀ JP! TẢI FILE NGAY DƯỚI ĐÂY:
👉 Tải TF:


/kaggle/working/models/best_model_TF.h5

👉 Tải JP:


/kaggle/working/models/best_model_JP.h5

('models/roberta_tokenizer/tokenizer_config.json',
 'models/roberta_tokenizer/special_tokens_map.json',
 'models/roberta_tokenizer/vocab.json',
 'models/roberta_tokenizer/merges.txt',
 'models/roberta_tokenizer/added_tokens.json',
 'models/roberta_tokenizer/tokenizer.json')

Metric cặp TF và JP

In [26]:
# [BLOCK ĐÁNH GIÁ 2 TRỤC CÒN LẠI: TF & JP]
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import re
import gc # Import garbage collection để dọn RAM
from transformers import RobertaTokenizerFast, TFRobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# 1. CẤU HÌNH (Phải khớp với lúc Train)
MAX_LEN = 256
BATCH_SIZE = 32

# 2. CHUẨN BỊ DỮ LIỆU (NẾU MẤT BIẾN THÌ NẠP LẠI)
if 'df' not in globals() or 'smart_text' not in df.columns:
    print("⏳ Đang nạp lại dữ liệu (Backup)...")
    # Tìm file
    data_path = None
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if '500' in filename: data_path = os.path.join(dirname, filename); break
    
    df = pd.read_csv(data_path)
    tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
    
    def basic_clean(text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\.\S+', '', text)
        mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                      'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
        pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
        return re.sub(pattern, '', text).strip()

    def smart_truncate(text, max_words=230):
        words = text.split()
        if len(words) <= max_words: return text
        return " ".join(words[:100] + words[-130:])

    df['clean_text'] = df['posts'].apply(basic_clean)
    df['smart_text'] = df['clean_text'].apply(smart_truncate)
    print("✅ Dữ liệu đã sẵn sàng.")

# 3. HÀM DỰNG MODEL & DATASET (GIỮ NGUYÊN KIẾN TRÚC)
def build_roberta_model():
    input_ids = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input_ids')
    mask = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='attention_mask')
    roberta = TFRobertaModel.from_pretrained('roberta-base', use_safetensors=False)
    embeddings = roberta(input_ids, attention_mask=mask)[0]
    cls_token = embeddings[:, 0, :]
    x = tf.keras.layers.Dense(256, activation='relu')(cls_token)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs=[input_ids, mask], outputs=output)
    return model

def create_dataset(texts):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='tf')
    return tf.data.Dataset.from_tensor_slices(dict(encodings)).batch(BATCH_SIZE)

# 4. TÍNH TOÁN CHỈ SỐ CHO TF VÀ JP
# --- THAY ĐỔI Ở ĐÂY: CHỈ ĐÁNH GIÁ TF VÀ JP ---
axes_to_check = {
    'TF': ('F', 'T'),  # Thinking là 1, Feeling là 0 (hoặc ngược lại tùy lúc train, code sẽ tự khớp)
    'JP': ('J', 'P')   # Perceiving là 1, Judging là 0
}

print("\n" + "="*70)
print("📊 BÁO CÁO HIỆU SUẤT ROBERTA (TRỤC TF & JP)")
print("="*70)

results_summary = []

for axis, (char0, char1) in axes_to_check.items():
    model_path = f"models/best_model_{axis}.h5"
    
    if os.path.exists(model_path):
        print(f"\n🔍 Đang đánh giá trục: {axis} ({char0} vs {char1})...")
        
        # Tái tạo tập validation y hệt lúc train (random_state=42)
        df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)
        _, val_texts, _, val_labels = train_test_split(
            df['smart_text'].tolist(), df['label'].tolist(), test_size=0.1, random_state=42, stratify=df['label']
        )
        
        # Load Model
        print("   -> Đang load trọng số (Weights)...")
        model = build_roberta_model()
        model.load_weights(model_path)
        
        # Predict
        print("   -> Đang dự đoán trên tập Validation...")
        y_prob = model.predict(create_dataset(val_texts), verbose=0)
        y_pred = (y_prob > 0.5).astype(int).flatten()
        
        # Tính chỉ số
        acc = accuracy_score(val_labels, y_pred)
        mf1 = f1_score(val_labels, y_pred, average='macro')
        wf1 = f1_score(val_labels, y_pred, average='weighted')
        
        print(f"✅ KẾT QUẢ TRỤC {axis}:")
        print(f"   - Accuracy:    {acc:.4f}")
        print(f"   - Macro F1:    {mf1:.4f}  <-- Chỉ số quan trọng nhất")
        print(f"   - Weighted F1: {wf1:.4f}")
        
        results_summary.append({'Axis': axis, 'Accuracy': acc, 'Macro F1': mf1, 'Weighted F1': wf1})
        
        # Dọn dẹp RAM ngay lập tức
        del model
        tf.keras.backend.clear_session()
        gc.collect()
    else:
        print(f"❌ CẢNH BÁO: Không tìm thấy file {model_path}. Ông đã train xong trục này chưa?")

# In bảng tổng kết
print("\n" + "="*70)
print("🏆 TỔNG HỢP KẾT QUẢ (TF & JP)")
final_df = pd.DataFrame(results_summary)
pd.options.display.float_format = '{:.4f}'.format
print(final_df)
print("="*70)


📊 BÁO CÁO HIỆU SUẤT ROBERTA (TRỤC TF & JP)

🔍 Đang đánh giá trục: TF (F vs T)...
   -> Đang load trọng số (Weights)...


Some layers from the model checkpoint at roberta-base were not used when initializing TFRobertaModel: ['lm_head']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at roberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


   -> Đang dự đoán trên tập Validation...
✅ KẾT QUẢ TRỤC TF:
   - Accuracy:    0.8525
   - Macro F1:    0.8430  <-- Chỉ số quan trọng nhất
   - Weighted F1: 0.8547

🔍 Đang đánh giá trục: JP (J vs P)...
   -> Đang load trọng số (Weights)...


Some layers from the model checkpoint at roberta-base were not used when initializing TFRobertaModel: ['lm_head']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at roberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


   -> Đang dự đoán trên tập Validation...
✅ KẾT QUẢ TRỤC JP:
   - Accuracy:    0.7513
   - Macro F1:    0.7472  <-- Chỉ số quan trọng nhất
   - Weighted F1: 0.7524

🏆 TỔNG HỢP KẾT QUẢ (TF & JP)
  Axis  Accuracy  Macro F1  Weighted F1
0   TF    0.8525    0.8430       0.8547
1   JP    0.7513    0.7472       0.7524
